In [37]:
import pandas as pd
import os
import requests
from io import StringIO

In [ ]:
# Charger le fichier CSV contenant les années et les URLs
csv_file = "../data/histo-validations-reseau-ferre.csv"
data = pd.read_csv(csv_file, sep=';')

In [ ]:
# Filtrer les années supérieures à 2022
data = data[data['annee'] >= 2022]

# Initialiser une liste pour stocker les DataFrames
dataframes = []

# Parcourir chaque ligne du fichier CSV filtré
for index, row in data.iterrows():
    year = row['annee']
    url = row['reseau_ferre']
    
    # Télécharger le fichier ZIP depuis l'URL
    response = requests.get(url)
    if response.status_code == 200:
        zip_file_path = f"../data/data-rf-{year}.zip"
        
        # Sauvegarder le fichier ZIP localement
        with open(zip_file_path, 'wb') as f:
            f.write(response.content)
        
        # Extraire le contenu du fichier ZIP
        extracted_folder = f"../data/data-rf-{year}"
        if not os.path.exists(extracted_folder):  # Vérifier si le dossier existe déjà
            os.makedirs(extracted_folder, exist_ok=True)
            os.system(f"tar -xf {zip_file_path} -C {extracted_folder}")
        
        # Lire les fichiers .txt extraits
        for file_name in os.listdir(extracted_folder):
            if file_name.endswith('.txt'):
                file_path = os.path.join(extracted_folder, file_name)
                # Charger le fichier .txt dans un DataFrame
                df = pd.read_csv(file_path, sep='\t', encoding='utf-8')
                df['annee'] = year  # Ajouter une colonne pour l'année
                dataframes.append(df)
    else:
        print(f"Erreur lors du téléchargement pour l'année {year}: {response.status_code}")


In [ ]:
dataframes

In [ ]:
# Vérifier si la liste des DataFrames n'est pas vide
if dataframes:
	# Combiner tous les DataFrames en un seul
	final_dataframe = pd.concat(dataframes, ignore_index=True)

	# Afficher un aperçu du DataFrame final
	print(final_dataframe.head())
else:
	print("La liste des DataFrames est vide. Aucun fichier n'a été traité.")

In [ ]:
# Sauvegarder le DataFrame final dans un fichier CSV
final_dataframe.to_csv("../data/combined_data.csv", index=False, sep=';')

In [ ]:
# à avoir les sections en haut de la page

In [39]:
dataset = pd.read_csv("../data/valid-n-jour-1er-semestre.csv", sep=';')

In [41]:
dataset.head(5)

,jour,code_stif_trns,code_stif_res,code_stif_arret,libelle_arret,ida,categorie_titre,nb_vald
0,2024-01-22,100,110.0,691.0,PTE D.CHAPELLE,72064,Amethyste,292
1,2024-01-22,100,110.0,691.0,PTE D.CHAPELLE,72064,Autres titres,337
2,2024-01-22,100,110.0,691.0,PTE D.CHAPELLE,72064,Imagine R,1539
3,2024-01-22,100,110.0,692.0,PTE D.VILLETTE,72430,Contrat Solidarité Transport,2160
4,2024-01-22,100,110.0,693.0,PTE MONTREUIL,71710,Contrat Solidarité Transport,2277


In [42]:
# Calculer la somme de nb_vald par categorie_titre
if 'dataset' in locals():
    grouped_data = dataset.groupby('categorie_titre')['nb_vald'].sum().reset_index()

    # Afficher le résultat
    print(grouped_data)

    # Sauvegarder le résultat dans un fichier CSV si nécessaire
    grouped_data.to_csv("../data/nb_vald_par_categorie_titre.csv", index=False, sep=';')
else:
    print("Le DataFrame 'dataset' n'est pas défini.")

                categorie_titre    nb_vald
0                     Amethyste   20795674
1                 Autres titres   82016303
2  Contrat Solidarité Transport  120948494
3                Forfait Navigo  492335516
4               Forfaits courts    7377389
5                     Imagine R  198765739
6                    NON DEFINI   26054833


In [ ]:
# Calculer la somme de nb_vald par categorie_titre
if 'dataset' in locals():
    grouped_data = dataset.groupby('categorie_titre')['nb_vald'].sum().reset_index()

    # Formater les nombres avec des virgules pour la lisibilité
    grouped_data['nb_vald'] = grouped_data['nb_vald'].apply(lambda x: f"{x:,}".replace(",", " "))

    # Afficher le résultat
    print(grouped_data)

    # Sauvegarder le résultat dans un fichier CSV si nécessaire
    grouped_data.to_csv("../data/nb_vald_par_categorie_titre.csv", index=False, sep=';')
else:
    print("Le DataFrame 'dataset' n'est pas défini.")

                categorie_titre      nb_vald
0                     Amethyste   20 795 674
1                 Autres titres   82 016 303
2  Contrat Solidarité Transport  120 948 494
3                Forfait Navigo  492 335 516
4               Forfaits courts    7 377 389
5                     Imagine R  198 765 739
6                    NON DEFINI   26 054 833
